In [2]:
import os, sys, imaplib, pathlib
import json
from google.colab import drive
drive.mount('/content/drive')#打开左侧文件夹查看位置!

import torch
print(torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.10.0+cu128
cuda


In [3]:
from facenet_pytorch import MTCNN


In [4]:
# !pip install facenet-pytorch# 下载后重启

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [5]:
# !pip install --upgrade torch torchvision# 下载后重启

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 160.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## input_output

In [12]:
DATA_FOLDER="/content/drive/MyDrive/airbnbMM2/host_about/data_processed"
IMAGES_FOLDER="/content/drive/MyDrive/airbnbMM2/host_pic/images"

os.makedirs(IMAGES_FOLDER, exist_ok=True)

input_output={}
for folder in [ f for f in os.listdir(DATA_FOLDER) if not f.endswith('.csv')]:
    # folder = city+time

    # ---listings---
    files=os.listdir(os.path.join(DATA_FOLDER, folder))
    for f in files:
        if f.startswith('listings_processed'):
            path_listings=os.path.join(DATA_FOLDER, folder, f)
            path_listings_zsc=os.path.join(DATA_FOLDER, folder, f"listings_zsc_{folder}.csv")
    path_results_zsc=os.path.join(DATA_FOLDER, folder, f"resultsZSC_{folder}.csv")

    # ---images folder---
    pic_folder=os.path.join(IMAGES_FOLDER, folder)
    os.makedirs(pic_folder, exist_ok=True)

    # --resultsface---
    resultsFACE_folder=os.path.join(IMAGES_FOLDER, 'results')
    os.makedirs(resultsFACE_folder, exist_ok=True)

    path_results_face=os.path.join(resultsFACE_folder, f"face_{folder}.json")
    path_results_deepface=os.path.join(resultsFACE_folder, f"deepface_{folder}.json")

    # ---gather---
    input_output[folder]={"listings":path_listings,
              "results_zsc":path_results_zsc,
              "listings_zsc":path_listings_zsc,

              'pic_folder':pic_folder,
              'results_face':path_results_face,
              "results_deepface":path_results_deepface}

for k,v in input_output.items():
    print(f"{k}".center(100,'-'))
    path_listings=v['listings']
    path_results_zsc=v['results_zsc']
    path_listings_zsc=v['listings_zsc']

    pic_folder=v['pic_folder']
    path_results_face=v['results_face']
    path_results_deepface=v['results_deepface']
    print(path_listings)
    print(path_results_zsc)
    print(path_listings_zsc,"\n")
    print("pic folder:",pic_folder)
    print(path_results_face)
    print(path_results_deepface,"\n")

---------------------------------------------paris_2306---------------------------------------------
/content/drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2306/listings_processed_paris_2306_32901.csv
/content/drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2306/resultsZSC_paris_2306.csv
/content/drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2306/listings_zsc_paris_2306.csv 

pic folder: /content/drive/MyDrive/airbnbMM2/host_pic/images/paris_2306
/content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_paris_2306.json
/content/drive/MyDrive/airbnbMM2/host_pic/images/results/deepface_paris_2306.json 

--------------------------------------------london_2406---------------------------------------------
/content/drive/MyDrive/airbnbMM2/host_about/data_processed/london_2406/listings_processed_london_2406_49856.csv
/content/drive/MyDrive/airbnbMM2/host_about/data_processed/london_2406/resultsZSC_london_2406.csv
/content/drive/MyDrive/airbnbMM2/host_about/data

## def

In [5]:
import numpy as np
import cv2
import os
import time
import json
from PIL import Image
import torch
from transformers import CLIPModel, CLIPProcessor
from facenet_pytorch import MTCNN
from tqdm import tqdm

## requirements
# facenet-pytorch==2.6.0
# torch==2.10.0
# torchaudio==2.10.0
# torchvision==0.25.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
mtcnn = MTCNN(keep_all=True, device=device)

cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [14]:
def analyze_background_semantic(img_rgb, bbox_list):
    """
    通过life/pro标签，用clip检测背景类型。

    不考虑人脸，仅考虑背景，
    计算和life/pro组标签相似度的平均值哪个更高。
    """

    h, w, _ = img_rgb.shape
    mask = np.ones((h, w), dtype=np.uint8)

    # -- faces zone ---
    for (x1, y1, x2, y2) in bbox_list:
        mask[y1:y2, x1:x2] = 0

    # ---black face zones---
    bg_img = img_rgb.copy()
    bg_img[mask == 0] = 0


    # ---prompt---
    clean_prompts = [
        "a close-up headshot with a neutral wall background",
        "a portrait photo focused on a person with no other objects",
        "a professional studio profile picture"
    ]
    lifestyle_prompts = [
        "a person traveling outdoors",
        "a person on vacation in a city or nature scene",
        "a person doing sports or leisure activities",
        "a family or social gathering indoors or outdoors"
    ]
    prompts = clean_prompts + lifestyle_prompts


    # ---classify type of bg---
    bg_pil = Image.fromarray(bg_img)
    inputs = clip_processor(
        text=prompts,
        images=bg_pil,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        logits = outputs.logits_per_image
        probs = logits.softmax(dim=1).cpu().numpy()[0]

    clean_score = probs[:len(clean_prompts)].mean()
    lifestyle_score = probs[len(clean_prompts):].mean()

    is_lifestyle = lifestyle_score > clean_score

    return {
        "clean_score": float(clean_score),
        "lifestyle_score": float(lifestyle_score),
        "is_lifestyle_background": bool(is_lifestyle)
    }




def classify_pic_type(img_path):
    host_id = os.path.splitext(os.path.basename(img_path))[0]
    host_id=int(host_id)
    img=cv2.imread(img_path)

    # --init--
    # face：
    has_face, nb_face, face_area_ratio, avg_face_prob = 0, 0, 0, 0
    bbox_list=[]
    clean_score, lifestyle_score, host_picture_type=0, 0, "no_person"
    # deepface:
    age, age_class, gender, dominant_emotion=None,None,None,None
    smile_score, is_smiling=0,0


    if not img is None:# 若为none，不更新
        img_rgb=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _=img_rgb.shape
        img_area=h * w

        # ---face---
        boxes, probs = mtcnn.detect(img_rgb, landmarks=False)

        # 若检测到人脸, 覆盖初始化结果:
        if boxes is not None and len(boxes) > 0: # nb_face!=0
            has_face = 1
            nb_face = len(boxes)
            total_face_area = 0
            valid_probs = []

            for box, prob in zip(boxes, probs):
                if prob is None:
                    continue
                x1, y1, x2, y2 = box.astype(int) # 防止越界
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)
                area = (x2 - x1) * (y2 - y1)
                total_face_area += area

                bbox_list.append([int(x1), int(y1), int(x2), int(y2)])
                valid_probs.append(prob)
                face_area_ratio = total_face_area / img_area
                avg_face_prob = sum(valid_probs) / len(valid_probs) if valid_probs else 0

            if nb_face>1:
                host_picture_type='life_style'

            else :# nb_face==1
                bg_semantic = analyze_background_semantic(img_rgb, bbox_list)
                clean_score=bg_semantic['clean_score']
                lifestyle_score=bg_semantic['lifestyle_score']
                host_picture_type="pro_style" if clean_score > lifestyle_score else "life_style"

    # --- save ---
    res = {
        "img_path": img_path,
        "host_id": host_id,
        "has_face": has_face,
        "nb_face": nb_face,
        "face_area_ratio": face_area_ratio,
        "avg_face_prob": avg_face_prob,
        "bbox_list": bbox_list,
        "clean_score":clean_score,
        "lifestyle_score": lifestyle_score,
        "host_picture_type":host_picture_type,

        # deepface
        "age": age,
        "age_class": age_class,
        "gender": gender,
        "smile_score": smile_score,
        "is_smiling": is_smiling,
        "dominant_emotion": dominant_emotion
        }

    return res




def run_clf(pic_folder, path_results,save_interval=1000, n_sample=10):
    # ---detector---
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # mtcnn = MTCNN(keep_all=True, device=device)

    # ---no repetition---
    if os.path.exists(path_results):
        with open(path_results, 'r', encoding='utf-8')as f:
            results=json.load(f)

        processed=[int(result['host_id']) for result in results]
        pending_files=[os.path.join(pic_folder, f) for f in os.listdir(pic_folder) if f.endswith('.jpg') and os.path.splitext(f)[0].isdigit() and int(os.path.splitext(f)[0]) not in processed]

        print(f"[info] {len(processed)} / {len(os.listdir(pic_folder))}  already classified; {len(pending_files)} / {len(os.listdir(pic_folder))} pic to classify")
    else :
        results=[]
        pending_files=[os.path.join(pic_folder, f) for f in os.listdir(pic_folder) if f.endswith('.jpg')]
        print(f"[info] {len(pending_files)} / {len(os.listdir(pic_folder))} pic to classify")


    # ---n_sample---
    if n_sample:
        pending_files=pending_files[:n_sample]

    if len(pending_files)>0:

        # ---clf---
        start_time=time.time()
        for i, img_path in enumerate(tqdm(pending_files, total=len(pending_files), desc="processing images...")):
            # ---save interval---
            if (i+1) % save_interval==0:
                with open(path_results, 'w', encoding="utf-8")as f:
                    json.dump(results, f, indent=2, ensure_ascii=False)
                    print(f"\n[interval save] {i+1} / {len(pending_files)} results saved!\n")

            results.append(classify_pic_type(img_path))
        end_time=time.time()

        # ---final save---
        with open(path_results, 'w', encoding="utf-8")as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
            print(f"[final save] {len(results)} results saved to {path_results}\n")
        print(f"[DONE] {len(results)} images classified : {end_time-start_time:.2f} sec!\n")
    else :
        print(f"[info] all classified!\n")
    return results


In [15]:
# 2H(gpu)/8H(cpu) for 20K ?

for i, paths in input_output.items():
  print(f"{i}".center(100,'='))
  pic_folder=paths['pic_folder']
  path_results_face=paths['results_face']
  print(f"path_results:{path_results_face}")
  results=run_clf(pic_folder, path_results_face, save_interval=1000, n_sample=None)

=============================================paris_2306=============================================
path_results:/content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_paris_2306.json
[info] 21652 / 21652  already classified; 0 / 21652 pic to classify
[info] all classified!

============================================london_2406=============================================
path_results:/content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_london_2406.json
[info] 24885 / 24885  already classified; 0 / 24885 pic to classify
[info] all classified!

============================================london_2306=============================================
path_results:/content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_london_2306.json
[info] 21996 / 21996  already classified; 0 / 21996 pic to classify
[info] all classified!

=============================================paris_2406=============================================
path_results:/content/drive/MyDrive/airbnbMM2

processing images...: 100%|██████████| 19/19 [00:09<00:00,  1.99it/s]


[final save] 43074 results saved to /content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_paris_2406.json

[DONE] 43074 images classified : 9.55 sec!



## vis

In [24]:
import random
import cv2
import json
import matplotlib.pyplot as plt

path_result=input_output['paris_2406']['results_face']
print(path_result)

with open(path_result, 'r', encoding='utf-8')as f:
    results=json.load(f)


def visualize_face_results(results, n_samples=10, seed=42):
    """
    随机可视化若干检测结果
    """
    random.seed(seed)
    samples = random.sample(results, min(n_samples, len(results)))

    n_cols = 5
    n_rows = (len(samples) + n_cols - 1) // n_cols

    plt.figure(figsize=(4*n_cols, 4*n_rows))

    for i, item in enumerate(samples):
        img = cv2.imread(item["img_path"])
        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 画 bbox
        for bbox in item["bbox_list"]:
            x1, y1, x2, y2 = bbox
            cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (255,0,0), 2)

        plt.subplot(n_rows, n_cols, i+1)
        plt.imshow(img_rgb)
        plt.axis("off")


        title = (
            f"{item['host_id']}\n"
            f"has_face={item['has_face']} | nb={item['nb_face']}\n"
            # f"area_ratio={round(item['face_area_ratio'],3)}\n"
            # f"avg_prob={round(item['avg_face_prob'],3)}\n"
            f"clean_score={item['clean_score']:.2f} | lifestyle_scores={item['lifestyle_score']:.2f}\n"
            f"Type={item['host_picture_type']}"

        )
        plt.title(title, fontsize=9)

    plt.tight_layout()
    plt.show()


/content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_paris_2406.json


In [25]:
visualize_face_results(results, n_samples=50, seed=10)


<Figure size 2000x4000 with 0 Axes>